# 02 — Inferensi pretrained (DeepForest) lewat adapter API

Model: `weecology/deepforest-tree` — **kandidat tajuk pohon**, dilatih pada citra pesawat NEON ±10 cm/px.
Bukan model tanaman pertanian; belum diuji pada K919. Notebook memanggil `app.inference.DeepForestDetector`
(adapter yang sama dengan `POST /photos/{id}/analyze`), memuat model SEKALI dari folder lokal tanpa unduhan,
dan mengukur waktu sebenarnya di mesin ini.

Prasyarat (sekali): `apps\api\scripts\download_deepforest.py`. Lokasi bobot: env `WALLACEA_DEEPFOREST_DIR`
atau default `apps/api/var/models/deepforest-tree`.

In [ ]:
%matplotlib inline
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

import wallacea_ml
from wallacea_ml.prepare import ML_ROOT
print("wallacea_ml", wallacea_ml.__version__, "| ML_ROOT =", ML_ROOT)


def _df(rows):
    """Tabel: pandas bila terpasang (opsional), selain itu list apa adanya."""
    try:
        import pandas as pd
        return pd.DataFrame(rows)
    except ImportError:
        return rows


## 1. Identitas model & bobot

In [ ]:
from wallacea_ml import backend
model_dir = backend.default_deepforest_dir()
manifest = json.loads((model_dir / "wallacea_model.json").read_text(encoding="utf-8"))
print(json.dumps({k: manifest[k] for k in ("repo_id", "revision", "license", "model_version", "source")}, indent=1))
print({n: f["sha256"][:16] + "…" for n, f in {**manifest["files"], **manifest["aux_files"]}.items()})

## 2. Jalankan pada semua foto contoh (CPU). Waktu = median 3 pengulangan per foto.

In [ ]:
import platform, torch
from wallacea_ml.samples import load_samples
from wallacea_ml import pretrained
PARAMS = dict(confidence=0.35, device="cpu", patch_size=400, patch_overlap=0.05)   # = default API
det = backend.build_api_deepforest(model_dir, **PARAMS)
samples = load_samples()
result = pretrained.run_on_samples(det, samples, repeats=3)
out_path = pretrained.save(result)
print(f"mesin: {platform.processor() or platform.machine()} | torch {torch.__version__} | device: {PARAMS['device']}")
print(f"muat model: {result['load_seconds']} s (sekali)  →  {out_path.relative_to(ML_ROOT)}")
display(_df([{"foto": i["image"], "px": f"{i['width']}×{i['height']}", "mode": i["mode"],
              "detik (median)": i["seconds_median"], "detik (semua)": i["seconds"], "kandidat": len(i["pred"])}
             for i in result["images"]]))
print("parameter adapter:", result["parameters"])

## 3. Kandidat model (biru) — belum dibandingkan dengan anotasi manusia

In [ ]:
from wallacea_ml.viz import draw_boxes
fig, axes = plt.subplots(1, len(samples), figsize=(6 * len(samples), 6))
for ax, s, rec in zip(axes, samples, result["images"]):
    ax.imshow(draw_boxes(backend.load_oriented(s.path), [("pred", p["bbox"]) for p in rec["pred"]]))
    ax.set_title(f"{s.id}: {len(rec['pred'])} kandidat (pending)"); ax.axis("off")
plt.tight_layout(); plt.show()
conf = [p["confidence"] for r in result["images"] for p in r["pred"]]
plt.figure(figsize=(6, 3)); plt.hist(conf, bins=20); plt.xlabel("confidence (skor model, BUKAN akurasi)"); plt.show()
print("label asli model:", Counter(p["model_label"] for r in result["images"] for p in r["pred"]),
      "→ kind", Counter(p["label"] for r in result["images"] for p in r["pred"]))

## 4. Foto K919 — kualitatif saja

Tanpa anotasi manusia, yang dilaporkan hanya kandidat + waktu. Tidak ada angka akurasi.

In [ ]:
import time
K919 = ML_ROOT / "data" / "k919" / "raw"
ks = sorted(p for p in K919.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}) if K919.is_dir() else []
if not ks:
    print("Belum ada foto K919 — hasil notebook ini hanya untuk foto contoh NEON.")
for p in ks[:3]:
    img = backend.load_oriented(p)
    t = time.perf_counter(); out = det.detect(img); secs = time.perf_counter() - t
    plt.figure(figsize=(12, 8)); plt.imshow(draw_boxes(img, [("pred", d.bbox) for d in out.detections], max_side=2400))
    plt.title(f"{p.name} {img.size}: {len(out.detections)} kandidat tajuk pohon, {secs:.1f} s — TANPA ground truth")
    plt.axis("off"); plt.show()